In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report

DATA_PATH = "/Users/kasimozel/Desktop/Akıllı Tarım Asistanı/ml_service/data/"
MODEL_PATH = "/Users/kasimozel/Desktop/Akıllı Tarım Asistanı/ml_service/models/"

# Veriyi yukle
df = pd.read_csv(DATA_PATH + "normalize_veri.csv")

print("=" * 40)
print("VERI GENEL BILGI")
print("=" * 40)
print(f"Toplam satir: {len(df)}")
print(f"Sutunlar: {df.columns.tolist()}")
print(f"\nIlk 5 satir:")
print(df.head())

VERI GENEL BILGI
Toplam satir: 21920
Sutunlar: ['tarih', 'max_sicaklik', 'min_sicaklik', 'yagis', 'ruzgar_hizi', 'sehir', 'bolge', 'etiket', 'etiket_kod']

Ilk 5 satir:
        tarih  max_sicaklik  min_sicaklik     yagis  ruzgar_hizi  sehir  \
0  2022-01-01     -0.887149     -1.036406 -0.373763    -1.741860  Konya   
1  2022-01-02     -1.057262     -0.713490 -0.029617     0.312047  Konya   
2  2022-01-03     -1.113966     -1.448403 -0.373763    -0.961036  Konya   
3  2022-01-04     -0.792641     -1.025271 -0.373763    -1.079857  Konya   
4  2022-01-05     -0.669782     -0.958461 -0.373763    -0.451803  Konya   

        bolge etiket  etiket_kod  
0  Ic Anadolu  uygun           1  
1  Ic Anadolu  uygun           1  
2  Ic Anadolu  uygun           1  
3  Ic Anadolu  uygun           1  
4  Ic Anadolu  uygun           1  


In [2]:
# Model icin sadece sayisal sutunlari al
ozellikler = ["max_sicaklik", "min_sicaklik", "yagis", "ruzgar_hizi"]
X = df[ozellikler]

# Isolation Forest modelini olustur ve egit
# contamination → verideki anomali orani (bizim veride uygun_degil %3'tu)
model = IsolationForest(
    n_estimators=100,
    contamination=0.03,
    random_state=42
)

model.fit(X)

# Tahmin yap (1 = normal, -1 = anomali)
df["anomali_skoru"] = model.decision_function(X)
df["anomali"] = model.predict(X)
df["anomali_etiket"] = df["anomali"].map({1: "normal", -1: "anomali"})

print("=" * 40)
print("ISOLATION FOREST SONUCLARI")
print("=" * 40)
print(f"\nAnomali dagilimi:")
print(df["anomali_etiket"].value_counts())
print(f"\nYuzdelik:")
print(df["anomali_etiket"].value_counts(normalize=True).round(3) * 100)

ISOLATION FOREST SONUCLARI

Anomali dagilimi:
anomali_etiket
normal     21262
anomali      658
Name: count, dtype: int64

Yuzdelik:
anomali_etiket
normal     97.0
anomali     3.0
Name: proportion, dtype: float64


In [3]:
# Anomali olan gunleri incele
anomali_df = df[df["anomali_etiket"] == "anomali"].copy()

print("=" * 40)
print("ANOMALI OLAN GUNLER")
print("=" * 40)
print(f"Toplam anomali sayisi: {len(anomali_df)}")

print(f"\nSehire gore anomali dagilimi:")
print(anomali_df["sehir"].value_counts())

print(f"\nAnomali gunlerin hava degerleri (orijinal veri):")
orijinal_df = pd.read_csv(DATA_PATH + "islenmis_veri.csv")
anomali_orijinal = orijinal_df[df["anomali_etiket"] == "anomali"]
print(anomali_orijinal[["tarih", "sehir", "max_sicaklik", 
                          "min_sicaklik", "yagis", 
                          "ruzgar_hizi", "etiket"]].head(10))

ANOMALI OLAN GUNLER
Toplam anomali sayisi: 658

Sehire gore anomali dagilimi:
sehir
Erzurum       71
Bitlis        65
Adana         47
Samsun        46
Sivas         45
Izmir         45
Tekirdag      40
Edirne        40
Agri          29
Sanliurfa     27
Gaziantep     27
Tokat         26
Malatya       24
Diyarbakir    23
Van           22
Afyon         22
Konya         21
Kayseri       17
Ankara        12
Eskisehir      9
Name: count, dtype: int64

Anomali gunlerin hava degerleri (orijinal veri):
         tarih  sehir  max_sicaklik  min_sicaklik  yagis  ruzgar_hizi  \
18  2022-01-19  Konya          -2.2          -7.1    0.6         28.1   
22  2022-01-23  Konya          -0.8          -8.3   22.2         18.3   
24  2022-01-25  Konya          -6.6         -11.5   11.8         12.1   
25  2022-01-26  Konya          -6.2          -9.1    3.5         15.8   
27  2022-01-28  Konya         -10.0         -17.2    0.0          8.7   
28  2022-01-29  Konya          -8.3         -13.7    0.0      

In [4]:
# Sahte anormal veri ile test et
scaler = joblib.load(MODEL_PATH + "scaler.pkl")

test_verisi = pd.DataFrame([
    # Normal bir gun
    {"max_sicaklik": 22.0, "min_sicaklik": 10.0, 
     "yagis": 5.0, "ruzgar_hizi": 15.0},
    # Anomali - asiri soguk
    {"max_sicaklik": -25.0, "min_sicaklik": -40.0, 
     "yagis": 0.0, "ruzgar_hizi": 10.0},
    # Anomali - asiri yagis
    {"max_sicaklik": 18.0, "min_sicaklik": 8.0,  
     "yagis": 120.0, "ruzgar_hizi": 20.0},
    # Anomali - asiri sicak
    {"max_sicaklik": 45.0, "min_sicaklik": 30.0, 
     "yagis": 0.0, "ruzgar_hizi": 5.0},
])

# Normalize et
test_normalize = scaler.transform(test_verisi)

# Tahmin yap
tahminler = model.predict(test_normalize)
skorlar = model.decision_function(test_normalize)

print("=" * 40)
print("MODEL TEST SONUCLARI")
print("=" * 40)
for i, (tahmin, skor) in enumerate(zip(tahminler, skorlar)):
    durum = "NORMAL" if tahmin == 1 else "ANOMALI"
    print(f"Test {i+1}: {durum} (skor: {skor:.3f})")
    print(f"  Veri: {test_verisi.iloc[i].to_dict()}")
    print()

# Modeli kaydet
joblib.dump(model, MODEL_PATH + "isolation_forest.pkl")
print("Model kaydedildi → models/isolation_forest.pkl")

MODEL TEST SONUCLARI
Test 1: NORMAL (skor: 0.151)
  Veri: {'max_sicaklik': 22.0, 'min_sicaklik': 10.0, 'yagis': 5.0, 'ruzgar_hizi': 15.0}

Test 2: ANOMALI (skor: -0.032)
  Veri: {'max_sicaklik': -25.0, 'min_sicaklik': -40.0, 'yagis': 0.0, 'ruzgar_hizi': 10.0}

Test 3: ANOMALI (skor: -0.068)
  Veri: {'max_sicaklik': 18.0, 'min_sicaklik': 8.0, 'yagis': 120.0, 'ruzgar_hizi': 20.0}

Test 4: ANOMALI (skor: -0.044)
  Veri: {'max_sicaklik': 45.0, 'min_sicaklik': 30.0, 'yagis': 0.0, 'ruzgar_hizi': 5.0}

Model kaydedildi → models/isolation_forest.pkl


/Users/kasimozel/Desktop/Akıllı Tarım Asistanı/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(
/Users/kasimozel/Desktop/Akıllı Tarım Asistanı/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(


In [5]:
# Uyariyi duzelt - test verisini DataFrame olarak ver
test_normalize_df = pd.DataFrame(
    test_normalize, 
    columns=["max_sicaklik", "min_sicaklik", "yagis", "ruzgar_hizi"]
)

tahminler = model.predict(test_normalize_df)
skorlar = model.decision_function(test_normalize_df)

print("=" * 40)
print("DUZELTILMIS TEST SONUCLARI")
print("=" * 40)
for i, (tahmin, skor) in enumerate(zip(tahminler, skorlar)):
    durum = "NORMAL" if tahmin == 1 else "ANOMALI"
    print(f"Test {i+1}: {durum} (skor: {skor:.3f})")

print("\nUyari giderildi, model hazir!")


DUZELTILMIS TEST SONUCLARI
Test 1: NORMAL (skor: 0.151)
Test 2: ANOMALI (skor: -0.032)
Test 3: ANOMALI (skor: -0.068)
Test 4: ANOMALI (skor: -0.044)

Uyari giderildi, model hazir!
